In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report


In [ ]:
df = pd.read_csv("diabetes.csv")
print("Dataset Loaded Successfully ")
print("Dataset Shape:\n")
print(df.shape)
df.head()


In [ ]:
print("\nDataset Info:")
print(df.info())

print("\nChecking for missing values:\n", df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.model_selection import cross_val_score

k_values = range(1, 31)
cv_scores = []

for k in k_values:
    knn_cv = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Find the best K
best_k = k_values[np.argmax(cv_scores)]
best_score = max(cv_scores)

print(f"\n Best K value: {best_k}")
print(f" Cross-validated Accuracy: {best_score*100:.2f}%")

# Plot
plt.figure(figsize=(8,5))
plt.plot(k_values, cv_scores, marker='o', color='green')
plt.plot(best_k,best_score,'ro')
plt.title('Cross-Validation Accuracy vs K')
plt.xlabel('K value')
plt.ylabel('CV Accuracy')
plt.grid(True)
plt.show()


In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train_scaled, y_train)

In [ ]:
y_pred = knn.predict(X_test_scaled)


In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("\n===== MODEL EVALUATION =====")
print(f"Accuracy     : {accuracy*100:.2f}%")
print(f"Error Rate   : {error_rate*100:.2f}%")
print(f"Precision    : {precision*100:.2f}%")
print(f"Recall       : {recall*100:.2f}%")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', linewidths=1)
plt.title(f'Confusion Matrix (K={best_k})')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
error_rates = []
for i in range(1, 21):
    knn_i = KNeighborsClassifier(n_neighbors=i)
    knn_i.fit(X_train_scaled, y_train)
    pred_i = knn_i.predict(X_test_scaled)
    error_rates.append(np.mean(pred_i != y_test))

plt.figure(figsize=(8,5))
plt.plot(range(1, 21), error_rates, color='red', linestyle='dashed', marker='o', markerfacecolor='blue')
plt.title('Error Rate vs K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Error')
plt.show()